## 1. Funcoes

In [0]:
/* =====================================================================
   SECAO 1 - FUNCOES
   ---------------------------------------------------------------------
   Funcoes SQL (Unity Catalog) reutilizadas por TODAS as tabelas Silver.
   Rode estas celulas UMA vez antes das tabelas.

   POR QUE FUNCOES?
   Cada regra de limpeza (texto, datas, Sim/Nao, duracao) aparece em varias
   colunas e varias tabelas. Escrevendo a regra em um unico lugar, uma
   correcao futura e feita em 1 ponto so, e as queries das tabelas ficam
   curtas e legiveis.

   ===================================================================== */

In [0]:
-- ---------------------------------------------------------------------
-- 1) fn_limpa_texto
--    Regra: a Bronze traz "valor vazio" de 4 formas diferentes:
--           NULL real, '' / ' ' (so espacos), o texto 'None' (artefato do
--           pandas/CSV) e, as vezes, 'null' / 'nan'.
--           Tudo isso considerado "nao informado" -> vira NULL.
--    Tambem: remove espacos nas pontas e colapsa espacos/quebras de linha
--           repetidos (a leitura do CSV usa multiLine=true, entao textos
--           livres podem trazer \n dentro do campo).
-- ---------------------------------------------------------------------
CREATE OR REPLACE FUNCTION anvisa.fn_limpa_texto(valor STRING)
RETURNS STRING
COMMENT 'Converte vazio/None/null/nan em NULL; trim e colapso de espacos.'
RETURN CASE
         WHEN lower(trim(regexp_replace(valor, '[ \t\r\n]+', ' '))) IN ('', 'none', 'null', 'nan')
           THEN NULL
         ELSE trim(regexp_replace(valor, '[ \t\r\n]+', ' '))
       END;

In [0]:
-- ---------------------------------------------------------------------
-- 2) fn_sim_nao
--    Regra: 'Sim' -> TRUE, 'Nao' -> FALSE, qualquer outra coisa -> NULL.
--    NULL (e nao FALSE) para "desconhecido": em farmacovigilancia, ausencia
--    de informacao NAO e o mesmo que resposta negativa.
--    Cobre 'Não ' com espaco sobrando e grafia sem acento.
-- ---------------------------------------------------------------------
CREATE OR REPLACE FUNCTION anvisa.fn_sim_nao(valor STRING)
RETURNS BOOLEAN
COMMENT 'Sim/Nao -> BOOLEAN; qualquer outro valor -> NULL.'
RETURN CASE
         WHEN lower(trim(valor)) IN ('sim', 's', 'yes', 'y', 'true') THEN TRUE
         WHEN lower(trim(valor)) IN ('não', 'nao', 'n', 'no', 'false') THEN FALSE
       END;

In [0]:
-- ---------------------------------------------------------------------
-- 3) fn_lista_pipe
--    Regra: a Bronze guarda listas no mesmo campo separadas por '|'
--           (ex.: 'Tamisa|Aerolin|CoronaVac'). Viram ARRAY<STRING>,
--           sem elementos vazios nem 'None'.
-- ---------------------------------------------------------------------
CREATE OR REPLACE FUNCTION anvisa.fn_lista_pipe(valor STRING)
RETURNS ARRAY<STRING>
COMMENT 'Texto separado por | -> ARRAY<STRING> sem vazios/None.'
RETURN filter(
         transform(split(valor, '[|]'), x -> trim(x)),
         x -> x <> '' AND lower(x) <> 'none'
       );

In [0]:
-- ---------------------------------------------------------------------
-- 4) fn_normaliza_data   (passo 1 do tratamento de datas)
--    Objetivo: reduzir os varios "jeitos de escrever" a UM formato de
--    trabalho, para o parser ficar simples.
--    - Pega so o trecho inicial numerico (descarta a hora):
--        '09/28/2023 00:00:00.000000' -> '09/28/2023'
--        '2023-09-28T10:00:00'        -> '2023-09-28'
--    - Troca '-' por '/' :  '2023-09-28' -> '2023/09/28'
--    - Texto sem numero ('None', '') -> NULL
-- ---------------------------------------------------------------------
CREATE OR REPLACE FUNCTION anvisa.fn_normaliza_data(valor STRING)
RETURNS STRING
COMMENT 'Extrai o trecho de data (sem hora) e padroniza o separador para /.'
RETURN NULLIF(
         regexp_replace(
           regexp_extract(trim(valor), '^([0-9]+(?:[/-][0-9]+)*)', 1),
           '-', '/'),
         '');

In [0]:
-- ---------------------------------------------------------------------
-- 5) fn_parse_data   (passo 2: converte para DATE, formato yyyy-MM-dd)
--
--    Parametro prioridade_br:
--      Datas como 03/04/2023 sao AMBIGUAS (3 de abril? 4 de marco?).
--      Quando o dia e <= 12 os dois formatos sao validos, e so o contexto
--      decide. Entao a ordem de tentativa e escolhida POR COLUNA:
--        TRUE  -> tenta dd/MM/yyyy primeiro, depois MM/dd/yyyy
--        FALSE -> tenta MM/dd/yyyy primeiro, depois dd/MM/yyyy
--      Quando o dia e > 12 (ex.: 28/09/2023) so um formato e valido e o
--      resultado e correto independentemente da prioridade.
--
--    Formatos aceitos (apos fn_normaliza_data) e regra para datas INCOMPLETAS:
--      yyyy/M/d            -> data completa (ISO)
--      yyyyMMdd            -> data completa (formato do VigiMed); tambem
--                             tenta ddMMyyyy se o primeiro falhar
--      yyyyMMddHHmm[ss]    -> usa so os 8 primeiros digitos
--      d/M/yyyy ou M/d/yyyy-> data completa (BR ou EUA, ver prioridade_br)
--      yyyy/M              -> SEM DIA   -> assume DIA 01 do mes
--      M/yyyy              -> SEM DIA   -> assume DIA 01 do mes
--      MMyyyy / yyyyMM     -> SEM DIA   -> assume DIA 01 (6 digitos: se os 4
--                             primeiros formam um ano 1900-2100 e yyyyMM,
--                             senao MMyyyy. Ex.: '042029' -> 2029-04-01)
--      yyyy                -> SEM DIA/MES -> assume 01 de janeiro
--      d/M (sem ano)       -> NULL: nao existe ano para "chutar". Inventar
--                             o ano geraria dado clinico falso.
--      qualquer outra coisa / data impossivel (31/02/2020) -> NULL
--
--    Sanidade: resultado fora de 1900-01-01..2100-12-31 vira NULL.
--
--    Estrutura: o SELECT interno calcula a data (d); o externo so aplica
--    a sanidade. try_to_date devolve NULL (nao quebra) se nao converter.
-- ---------------------------------------------------------------------
CREATE OR REPLACE FUNCTION anvisa.fn_parse_data(valor STRING, prioridade_br BOOLEAN)
RETURNS DATE
COMMENT 'Texto com formatos mistos -> DATE. Incompleta: dia 01 (mes) / 01-jan (ano). Sem ano -> NULL.'
RETURN (
  SELECT CASE WHEN d BETWEEN DATE'1900-01-01' AND DATE'2100-12-31' THEN d END
  FROM (
    SELECT CASE
             -- ISO: 2023-09-28 (ja normalizado para 2023/09/28)
             WHEN regexp_like(t, '^[0-9]{4}/[0-9]{1,2}/[0-9]{1,2}$')
               THEN try_to_date(t, 'yyyy/M/d')

             -- Compacto: 20230928 (padrao VigiMed). Plano B: 28092023
             WHEN regexp_like(t, '^[0-9]{8}$')
               THEN coalesce(try_to_date(t, 'yyyyMMdd'), try_to_date(t, 'ddMMyyyy'))

             -- Compacto com hora: 202309281030 / 20230928103000 -> usa 8 primeiros digitos
             WHEN regexp_like(t, '^[0-9]{12}([0-9]{2})?$')
               THEN try_to_date(substr(t, 1, 8), 'yyyyMMdd')

             -- Barras com ano no final: ambiguo BR x EUA -> depende da prioridade
             WHEN regexp_like(t, '^[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}$')
               THEN CASE
                      WHEN prioridade_br
                        THEN coalesce(try_to_date(t, 'd/M/yyyy'), try_to_date(t, 'M/d/yyyy'))
                      ELSE coalesce(try_to_date(t, 'M/d/yyyy'), try_to_date(t, 'd/M/yyyy'))
                    END

             -- Incompletas (sem dia) -> dia 01
             WHEN regexp_like(t, '^[0-9]{4}/[0-9]{1,2}$')            -- yyyy/M
               THEN try_to_date(concat(t, '/1'), 'yyyy/M/d')
             WHEN regexp_like(t, '^[0-9]{1,2}/[0-9]{4}$')            -- M/yyyy
               THEN try_to_date(concat('1/', t), 'd/M/yyyy')
             WHEN regexp_like(t, '^[0-9]{6}$')                       -- yyyyMM ou MMyyyy
               THEN CASE
                      WHEN substr(t, 1, 4) BETWEEN '1900' AND '2100'
                        THEN try_to_date(concat(t, '01'), 'yyyyMMdd')
                      ELSE try_to_date(concat('01', t), 'ddMMyyyy')
                    END

             -- Incompleta (so ano) -> 01 de janeiro
             WHEN regexp_like(t, '^[0-9]{4}$')
               THEN try_to_date(concat(t, '0101'), 'yyyyMMdd')
           END AS d
    FROM (SELECT anvisa.fn_normaliza_data(valor) AS t)
  )
);

In [0]:
-- ---------------------------------------------------------------------
-- 6) fn_data_incompleta
--    TRUE  = a data original NAO tinha o dia (foi completada com dia 01)
--    FALSE = data completa
--    NULL  = data ausente ou nao interpretavel
--    Serve para o analista saber quais datas sao aproximadas (ex.: nao
--    calcular "tempo ate o evento em dias" com datas aproximadas).
-- ---------------------------------------------------------------------
CREATE OR REPLACE FUNCTION anvisa.fn_data_incompleta(valor STRING)
RETURNS BOOLEAN
COMMENT 'TRUE se a data original nao tinha dia (completada com 01).'
RETURN CASE
         WHEN anvisa.fn_parse_data(valor, TRUE) IS NULL THEN NULL
         WHEN regexp_like(
                anvisa.fn_normaliza_data(valor),
                '^([0-9]{4}|[0-9]{6}|[0-9]{4}/[0-9]{1,2}|[0-9]{1,2}/[0-9]{4})$')
           THEN TRUE
         ELSE FALSE
       END;

In [0]:
-- ---------------------------------------------------------------------
-- 7) fn_duracao_em_dias
--    Entrada tipica do VigiMed: '5 minuto', '3 dia', '12 dia'.
--    Converte para DIAS (numero unico, comparavel entre registros).
--    Conversoes APROXIMADAS: mes = 30 dias, ano = 365 dias.
--    Texto que nao comeca com numero, ou unidade desconhecida -> NULL.
-- ---------------------------------------------------------------------
CREATE OR REPLACE FUNCTION anvisa.fn_duracao_em_dias(texto STRING)
RETURNS DECIMAL(18,4)
COMMENT 'Duracao textual (ex.: 5 minuto, 3 dia) -> dias (mes=30, ano=365).'
RETURN (
  SELECT CAST(
           CASE
             WHEN v IS NULL            THEN NULL
             WHEN u LIKE 'seg%'        THEN v / 86400
             WHEN u LIKE 'min%'        THEN v / 1440
             WHEN u LIKE 'hora%'       THEN v / 24
             WHEN u LIKE 'dia%'        THEN v
             WHEN u LIKE 'sem%'        THEN v * 7
             WHEN regexp_like(u, '^m[eê]s')  THEN v * 30
             WHEN u LIKE 'ano%'        THEN v * 365
           END AS DECIMAL(18,4))
  FROM (
    SELECT
      try_cast(replace(regexp_extract(trim(texto), '^([0-9]+([.,][0-9]+)?)', 1), ',', '.') AS DECIMAL(18,4)) AS v,
      lower(trim(regexp_extract(trim(texto), '^[0-9]+([.,][0-9]+)? *(.*)$', 2)))                              AS u
  )
);

## 2. Tabelas Silver

### 2.1 s_vigimed_notificacoes

In [0]:
/* =====================================================================
   2.1 - s_vigimed_notificacoes
   Origem : anvisa.b_vigimed_notificacoes   (Bronze, tudo STRING)
   Destino: anvisa.s_vigimed_notificacoes   (Silver, tipada)
   GRAO   : 1 linha por notificacao (id_notificacao unico)

   Etapas (CTEs), na ordem em que executam:
     1. bronze_valido -> descarta linhas "quebradas" do CSV
     2. limpo         -> NULL real no lugar de 'None'/vazio (so texto)
     3. tipado        -> converte tipos e datas
     4. deduplicado   -> 1 linha por notificacao (a mais recente/completa)
     5. final         -> regras de plausibilidade (valor absurdo vira NULL)
   ===================================================================== */

CREATE OR REPLACE TABLE anvisa.s_vigimed_notificacoes
USING DELTA
-- LOCATION 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/02_silver/s_vigimed_notificacoes'
COMMENT 'Silver VigiMed: 1 linha por notificacao, tipada e deduplicada. Detalhes de reacoes e medicamentos estao em s_vigimed_reacoes e s_vigimed_medicamentos.'
AS
WITH

/* ---------------------------------------------------------------------
   1. bronze_valido
   REGRA: 33.246 linhas da Bronze tem IDENTIFICACAO_NOTIFICACAO nulo e TODAS
   as outras colunas nulas (ex.: UF = 'Outro profissional de saude'). Sao
   fragmentos gerados por quebras de linha dentro de campos do CSV (multiLine),
   e nao notificacoes reais. Sem o identificador, a linha nao pode ser ligada a
   reacoes/medicamentos -> DESCARTADA.
   Mantemos so ids no padrao 'BR-ANVISA-<numeros>'.
   --------------------------------------------------------------------- */
bronze_valido AS (
  SELECT *
  FROM anvisa.b_vigimed_notificacoes
  WHERE regexp_like(upper(trim(IDENTIFICACAO_NOTIFICACAO)), '^BR-ANVISA-[0-9]+$')
),

/* ---------------------------------------------------------------------
   2. limpo
   REGRA: 'None', '' e ' ' viram NULL (fn_limpa_texto). Nenhum valor e
   inventado/imputado: em farmacovigilancia, "nao informado" deve continuar
   "nao informado".
   Os nomes das colunas continuam iguais aos da Bronze (MAIUSCULAS) para
   facilitar a rastreabilidade.
   --------------------------------------------------------------------- */
limpo AS (
  SELECT
    upper(trim(IDENTIFICACAO_NOTIFICACAO))                 AS IDENTIFICACAO_NOTIFICACAO,
    upper(anvisa.fn_limpa_texto(UF))                       AS UF,
    anvisa.fn_limpa_texto(TIPO_ENTRADA_VIGIMED)            AS TIPO_ENTRADA_VIGIMED,
    anvisa.fn_limpa_texto(RECEBIDO_DE)                     AS RECEBIDO_DE,
    anvisa.fn_limpa_texto(TIPO_NOTIFICACAO)                AS TIPO_NOTIFICACAO,
    anvisa.fn_limpa_texto(NOTIFICACAO_PARENT_CHILD)        AS NOTIFICACAO_PARENT_CHILD,
    anvisa.fn_limpa_texto(GRUPO_IDADE)                     AS GRUPO_IDADE,
    anvisa.fn_limpa_texto(IDADE_GESTACIONAL_MOMENTO_REACAO) AS IDADE_GESTACIONAL_MOMENTO_REACAO,
    anvisa.fn_limpa_texto(SEXO)                            AS SEXO,
    anvisa.fn_limpa_texto(NOTIFICADOR)                     AS NOTIFICADOR,
    anvisa.fn_limpa_texto(GRAVE)                           AS GRAVE,
    -- Colunas abaixo sao numericas/data/booleanas: passam cruas e sao
    -- tratadas na etapa 'tipado'
    DATA_INCLUSAO_SISTEMA, DATA_ULTIMA_ATUALIZACAO, DATA_NOTIFICACAO, DATA_NASCIMENTO,
    IDADE_MOMENTO_REACAO, PESO_KG, ALTURA_CM, GESTANTE, LACTANTE,
    _data_ingestao
  FROM bronze_valido
),

/* ---------------------------------------------------------------------
   3. tipado
   REGRAS DE TIPAGEM
   - Datas de SISTEMA (inclusao, atualizacao, notificacao, nascimento) vem
     como 'MM/dd/yyyy HH:mm:ss.SSSSSS' (padrao EUA) -> prioridade_br = FALSE.
     Datas com dia > 12 sao resolvidas corretamente de qualquer forma.
   - Hora e descartada: a granularidade util aqui e o dia.
   - Gestante/Lactante: 'Sim'/'Nao' -> BOOLEAN (desconhecido = NULL).
   - GRAVE: no nivel da notificacao pode vir mais de um valor separado por
     '|' (1 por reacao). ind_grave = TRUE se QUALQUER reacao for grave;
     FALSE se so houver 'Nao'; NULL se nao informado.
   - Numericos: try_cast devolve NULL em vez de quebrar a pipeline.
   --------------------------------------------------------------------- */
tipado AS (
  SELECT
    IDENTIFICACAO_NOTIFICACAO                                         AS id_notificacao,
    -- UF: so aceita as 27 siglas validas; qualquer outro texto vira NULL
    CASE WHEN UF IN ('AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
                     'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO')
         THEN UF END                                                  AS uf,
    TIPO_ENTRADA_VIGIMED                                              AS tipo_entrada_vigimed,
    RECEBIDO_DE                                                       AS recebido_de,
    TIPO_NOTIFICACAO                                                  AS tipo_notificacao,
    NOTIFICACAO_PARENT_CHILD                                          AS notificacao_parent_child,

    anvisa.fn_parse_data(DATA_INCLUSAO_SISTEMA,    FALSE)             AS data_inclusao_sistema,
    anvisa.fn_parse_data(DATA_ULTIMA_ATUALIZACAO,  FALSE)             AS data_ultima_atualizacao,
    anvisa.fn_parse_data(DATA_NOTIFICACAO,         FALSE)             AS data_notificacao,
    anvisa.fn_parse_data(DATA_NASCIMENTO,          FALSE)             AS data_nascimento,

    try_cast(IDADE_MOMENTO_REACAO AS DECIMAL(6,2))                    AS idade_momento_reacao,  -- anos (premissa)
    GRUPO_IDADE                                                       AS grupo_idade,
    IDADE_GESTACIONAL_MOMENTO_REACAO                                  AS idade_gestacional_momento_reacao,
    SEXO                                                              AS sexo,
    anvisa.fn_sim_nao(GESTANTE)                                       AS ind_gestante,
    anvisa.fn_sim_nao(LACTANTE)                                       AS ind_lactante,
    try_cast(PESO_KG   AS DECIMAL(7,2))                               AS peso_kg,
    try_cast(ALTURA_CM AS DECIMAL(6,1))                               AS altura_cm,

    CASE WHEN GRAVE RLIKE '(?i)sim'        THEN TRUE
         WHEN GRAVE RLIKE '(?i)n[ãa]o'     THEN FALSE
    END                                                               AS ind_grave,
    NOTIFICADOR                                                       AS notificador,
    _data_ingestao                                                    AS dt_ingestao_bronze
  FROM limpo
),

/* ---------------------------------------------------------------------
   4. deduplicado
   REGRA: se o mesmo id_notificacao aparecer mais de uma vez, fica a versao
   MAIS RECENTE (maior data_ultima_atualizacao; depois maior
   data_inclusao_sistema). Se ainda empatar, fica a mais completa
   (peso/altura preenchidos). Notificacoes sao atualizadas ao longo do tempo
   (follow-up), entao a versao mais nova e a mais fiel.
   --------------------------------------------------------------------- */
deduplicado AS (
  SELECT *
  FROM tipado
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY id_notificacao
            ORDER BY data_ultima_atualizacao DESC NULLS LAST,
                     data_inclusao_sistema   DESC NULLS LAST,
                     peso_kg                 DESC NULLS LAST,
                     altura_cm               DESC NULLS LAST
          ) = 1
),

/* ---------------------------------------------------------------------
   5. final  -  REGRAS DE PLAUSIBILIDADE
   Valor impossivel NAO e corrigido nem apagado a linha: so a coluna vira
   NULL (a notificacao continua valendo para as demais analises).
   - peso_kg / altura_cm = 0  -> a origem usa 0 como "nao informado"
                                 (ex.: '0.000000000000000'). Vira NULL.
   - peso fora de (0, 500] kg  e altura fora de [20, 260] cm -> NULL
   - idade fora de [0, 120] anos -> NULL
   - datas de sistema no futuro -> NULL
   - data_nascimento posterior a data de inclusao (ou no futuro) -> NULL
   --------------------------------------------------------------------- */
final AS (
  SELECT
    id_notificacao,
    uf,
    tipo_entrada_vigimed,
    recebido_de,
    tipo_notificacao,
    notificacao_parent_child,
    CASE WHEN data_inclusao_sistema   <= current_date() THEN data_inclusao_sistema   END AS data_inclusao_sistema,
    CASE WHEN data_ultima_atualizacao <= current_date() THEN data_ultima_atualizacao END AS data_ultima_atualizacao,
    CASE WHEN data_notificacao        <= current_date() THEN data_notificacao        END AS data_notificacao,
    CASE WHEN data_nascimento <= current_date()
          AND (data_inclusao_sistema IS NULL OR data_nascimento <= data_inclusao_sistema)
         THEN data_nascimento END                                                       AS data_nascimento,
    CASE WHEN idade_momento_reacao BETWEEN 0 AND 120 THEN idade_momento_reacao END      AS idade_momento_reacao,
    grupo_idade,
    idade_gestacional_momento_reacao,
    sexo,
    ind_gestante,
    ind_lactante,
    CASE WHEN peso_kg   > 0  AND peso_kg   <= 500 THEN peso_kg   END                    AS peso_kg,
    CASE WHEN altura_cm >= 20 AND altura_cm <= 260 THEN altura_cm END                   AS altura_cm,
    ind_grave,
    notificador,
    dt_ingestao_bronze,
    current_timestamp()                                                                 AS dt_processamento_silver
  FROM deduplicado
)

SELECT * FROM final;

num_affected_rows,num_inserted_rows


### 2.2 s_vigimed_reacoes

In [0]:
/* =====================================================================
   2.2 - s_vigimed_reacoes
   Origem : anvisa.b_vigimed_reacoes   (Bronze, tudo STRING)
   Destino: anvisa.s_vigimed_reacoes   (Silver, tipada)
   GRAO   : 1 linha por (notificacao x reacao adversa). Uma notificacao
            pode ter varias reacoes, entao id_notificacao NAO e unico aqui.
   Hierarquia MedDRA preservada: LLT -> PT -> HLT -> HLGT -> SOC.

   Etapas (CTEs): limpo -> tipado -> regras -> final
   ===================================================================== */

CREATE OR REPLACE TABLE anvisa.s_vigimed_reacoes
USING DELTA
-- LOCATION 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/02_silver/s_vigimed_reacoes'
COMMENT 'Silver VigiMed: reacoes adversas (MedDRA) por notificacao, tipadas e sem duplicatas exatas.'
AS
WITH

/* ---------------------------------------------------------------------
   1. limpo
   REGRA: 'None'/vazio -> NULL. Descarta linhas sem identificador valido
   (fragmentos de CSV) e linhas sem NENHUM termo de reacao (LLT e PT nulos),
   pois uma reacao sem descricao nao tem uso analitico.
   --------------------------------------------------------------------- */
limpo AS (
  SELECT
    upper(trim(IDENTIFICACAO_NOTIFICACAO))                AS IDENTIFICACAO_NOTIFICACAO,
    anvisa.fn_limpa_texto(REACAO_EVTO_ADVERSO_MEDDRA_LLT) AS LLT,
    anvisa.fn_limpa_texto(PT)                             AS PT,
    anvisa.fn_limpa_texto(HLT)                            AS HLT,
    anvisa.fn_limpa_texto(HLGT)                           AS HLGT,
    anvisa.fn_limpa_texto(SOC)                            AS SOC,
    DATA_INICIO_HORA,
    DATA_FINAL_HORA,
    anvisa.fn_limpa_texto(DURACAO)                        AS DURACAO,
    anvisa.fn_limpa_texto(GRAVE)                          AS GRAVE,
    anvisa.fn_limpa_texto(GRAVIDADE)                      AS GRAVIDADE,
    anvisa.fn_limpa_texto(DESFECHO)                       AS DESFECHO,
    _data_ingestao
  FROM anvisa.b_vigimed_reacoes
),

filtrado AS (
  SELECT *
  FROM limpo
  WHERE regexp_like(IDENTIFICACAO_NOTIFICACAO, '^BR-ANVISA-[0-9]+$')
    AND coalesce(LLT, PT) IS NOT NULL
),

/* ---------------------------------------------------------------------
   2. tipado
   REGRAS
   - Datas: formato predominante e yyyyMMdd; a funcao tambem entende ISO,
     dd/MM/yyyy, MM/dd/yyyy e datas incompletas. Para datas ambiguas
     (ex.: 03/04/2023) assume-se padrao BRASILEIRO (prioridade_br = TRUE),
     porque sao datas informadas pelo notificador no Brasil.
   - ind_*_incompleta = TRUE quando o dia nao existia e foi assumido dia 01.
   - GRAVE ('Sim'/'Nao') -> BOOLEAN; desconhecido = NULL.
   - GRAVIDADE pode listar varios criterios separados por virgula
     (ex.: 'Ameaca a vida, Hospitalizacao...') -> ARRAY<STRING>.
     Premissa: nenhum nome de criterio contem virgula.
   - DURACAO ('5 minuto', '3 dia') -> duracao_dias (aprox.) e texto original.
   - DESFECHO 'Desconhecido' e mantido como categoria (resposta explicita
     da fonte), diferente de NULL ("nao preenchido").
   --------------------------------------------------------------------- */
tipado AS (
  SELECT
    IDENTIFICACAO_NOTIFICACAO                              AS id_notificacao,
    LLT                                                    AS reacao_llt,
    PT                                                     AS reacao_pt,
    HLT                                                    AS reacao_hlt,
    HLGT                                                   AS reacao_hlgt,
    SOC                                                    AS reacao_soc,

    anvisa.fn_parse_data(DATA_INICIO_HORA, TRUE)           AS data_inicio,
    anvisa.fn_data_incompleta(DATA_INICIO_HORA)            AS ind_data_inicio_incompleta,
    anvisa.fn_parse_data(DATA_FINAL_HORA, TRUE)            AS data_fim,
    anvisa.fn_data_incompleta(DATA_FINAL_HORA)             AS ind_data_fim_incompleta,

    DURACAO                                                AS duracao_texto,
    anvisa.fn_duracao_em_dias(DURACAO)                     AS duracao_dias,

    anvisa.fn_sim_nao(GRAVE)                               AS ind_grave,
    transform(split(GRAVIDADE, ','), x -> trim(x))         AS gravidade_criterios,
    DESFECHO                                               AS desfecho,
    _data_ingestao                                         AS dt_ingestao_bronze
  FROM filtrado
),

/* ---------------------------------------------------------------------
   3. regras  -  PLAUSIBILIDADE
   - Inicio/fim de reacao no futuro e impossivel -> NULL.
   - data_fim anterior a data_inicio: NAO se apaga nenhuma das duas (nao da
     para saber qual esta errada); apenas sinaliza em
     ind_data_fim_anterior_inicio para o analista decidir.
   --------------------------------------------------------------------- */
regras AS (
  SELECT
    * EXCEPT (data_inicio, data_fim),
    CASE WHEN data_inicio <= current_date() THEN data_inicio END AS data_inicio,
    CASE WHEN data_fim    <= current_date() THEN data_fim    END AS data_fim
  FROM tipado
),

final AS (
  SELECT
    id_notificacao, reacao_llt, reacao_pt, reacao_hlt, reacao_hlgt, reacao_soc,
    data_inicio, ind_data_inicio_incompleta,
    data_fim,    ind_data_fim_incompleta,
    coalesce(data_fim < data_inicio, FALSE) AS ind_data_fim_anterior_inicio,
    duracao_texto, duracao_dias,
    ind_grave, gravidade_criterios, desfecho,
    dt_ingestao_bronze,
    current_timestamp() AS dt_processamento_silver
  FROM regras
)

/* ---------------------------------------------------------------------
   DEDUPLICACAO
   REGRA: remove apenas duplicatas EXATAS (todas as colunas iguais).
   Nao usamos (id_notificacao, reacao) como chave porque a mesma reacao
   pode ser registrada duas vezes com datas/desfechos diferentes, e isso e
   informacao clinica legitima. current_timestamp() e igual em todas as
   linhas, entao nao interfere no DISTINCT.
   --------------------------------------------------------------------- */
SELECT DISTINCT * FROM final;

num_affected_rows,num_inserted_rows


### 2.3 s_vigimed_medicamentos

In [0]:
/* =====================================================================
   2.3 - s_vigimed_medicamentos
   Origem : anvisa.b_vigimed_medicamentos   (Bronze, tudo STRING)
   Destino: anvisa.s_vigimed_medicamentos   (Silver, tipada)
   GRAO   : 1 linha por (notificacao x medicamento). id_notificacao NAO e
            unico (uma notificacao pode ter varios medicamentos).

   Etapas (CTEs): limpo -> filtrado -> tipado -> regras -> final
   ===================================================================== */

CREATE OR REPLACE TABLE anvisa.s_vigimed_medicamentos
USING DELTA
-- LOCATION 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/02_silver/s_vigimed_medicamentos'
COMMENT 'Silver VigiMed: medicamentos (WHODrug) por notificacao, tipados e sem duplicatas exatas.'
AS
WITH

/* ---------------------------------------------------------------------
   1. limpo
   REGRA: 'None', vazio e ' ' -> NULL em TODO texto. Ex.: FREQUENCIA_DOSE
   vem como ' ' e DETENTOR_REGISTRO como 'None'.
   --------------------------------------------------------------------- */
limpo AS (
  SELECT
    upper(trim(IDENTIFICACAO_NOTIFICACAO))                         AS IDENTIFICACAO_NOTIFICACAO,
    anvisa.fn_limpa_texto(RELACAO_MEDICAMENTO_EVENTO)              AS RELACAO_MEDICAMENTO_EVENTO,
    anvisa.fn_limpa_texto(NOME_MEDICAMENTO_WHODRUG)                AS NOME_MEDICAMENTO_WHODRUG,
    anvisa.fn_limpa_texto(PRINCIPIOS_ATIVOS_WHODRUG)               AS PRINCIPIOS_ATIVOS_WHODRUG,
    anvisa.fn_limpa_texto(CODIGO_ATC)                              AS CODIGO_ATC,
    anvisa.fn_limpa_texto(DETENTOR_REGISTRO)                       AS DETENTOR_REGISTRO,
    anvisa.fn_limpa_texto(CONCENTRACAO)                            AS CONCENTRACAO,
    anvisa.fn_limpa_texto(COMPONENTE_SUSPEITO)                     AS COMPONENTE_SUSPEITO,
    anvisa.fn_limpa_texto(ACAO_ADOTADA)                            AS ACAO_ADOTADA,
    anvisa.fn_limpa_texto(PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO) AS PROBLEMAS_ADICIONAIS,
    anvisa.fn_limpa_texto(INDICACAO_MEDDRA)                        AS INDICACAO_MEDDRA,
    anvisa.fn_limpa_texto(INDICACAO_RELATADA_NOTIFICADOR_INICIAL)  AS INDICACAO_RELATADA,
    anvisa.fn_limpa_texto(DOSE)                                    AS DOSE,
    anvisa.fn_limpa_texto(FREQUENCIA_DOSE)                         AS FREQUENCIA_DOSE,
    anvisa.fn_limpa_texto(POSOLOGIA)                               AS POSOLOGIA,
    anvisa.fn_limpa_texto(DURACAO)                                 AS DURACAO,
    INICIO_ADMINISTRACAO,
    FIM_ADMINISTRACAO,
    anvisa.fn_limpa_texto(FORMA_FARMACEUTICA)                      AS FORMA_FARMACEUTICA,
    anvisa.fn_limpa_texto(VIA_ADMINISTRACAO)                       AS VIA_ADMINISTRACAO,
    anvisa.fn_limpa_texto(VIA_ADMINISTRACAO_MAE_PAI)               AS VIA_ADMINISTRACAO_MAE_PAI,
    anvisa.fn_limpa_texto(NUMELO_LOTE)                             AS NUMERO_LOTE,   -- corrige typo da origem
    _data_ingestao
  FROM anvisa.b_vigimed_medicamentos
),

/* ---------------------------------------------------------------------
   2. filtrado
   REGRA: descarta linha sem identificador valido ou sem nome do
   medicamento (nao ha o que analisar).
   --------------------------------------------------------------------- */
filtrado AS (
  SELECT *
  FROM limpo
  WHERE regexp_like(IDENTIFICACAO_NOTIFICACAO, '^BR-ANVISA-[0-9]+$')
    AND NOME_MEDICAMENTO_WHODRUG IS NOT NULL
),

/* ---------------------------------------------------------------------
   3. tipado
   REGRAS
   - PRINCIPIOS_ATIVOS e CODIGO_ATC: listas separadas por '|' -> ARRAY.
   - DOSE ('1 gram (g)'): separada em numero (dose_valor) e unidade
     (dose_unidade); o texto original e mantido em dose_texto.
   - DURACAO: texto original + duracao_dias (aprox.).
   - Datas de administracao: mesma estrategia das reacoes (prioridade BR,
     dia 01 quando incompleta, flag *_incompleta).
   - FREQUENCIA_DOSE / POSOLOGIA: texto livre, mantidos como STRING (nao ha
     padrao confiavel para converter em numero).
   --------------------------------------------------------------------- */
tipado AS (
  SELECT
    IDENTIFICACAO_NOTIFICACAO                         AS id_notificacao,
    RELACAO_MEDICAMENTO_EVENTO                        AS relacao_medicamento_evento,
    NOME_MEDICAMENTO_WHODRUG                          AS nome_medicamento,
    anvisa.fn_lista_pipe(PRINCIPIOS_ATIVOS_WHODRUG)   AS principios_ativos,
    anvisa.fn_lista_pipe(CODIGO_ATC)                  AS codigos_atc,
    DETENTOR_REGISTRO                                 AS detentor_registro,
    CONCENTRACAO                                      AS concentracao,
    COMPONENTE_SUSPEITO                               AS componente_suspeito,
    ACAO_ADOTADA                                      AS acao_adotada,
    PROBLEMAS_ADICIONAIS                              AS problemas_adicionais,
    INDICACAO_MEDDRA                                  AS indicacao_meddra,
    INDICACAO_RELATADA                                AS indicacao_relatada,

    DOSE                                              AS dose_texto,
    try_cast(replace(regexp_extract(DOSE, '^([0-9]+([.,][0-9]+)?)', 1), ',', '.') AS DECIMAL(18,4)) AS dose_valor,
    NULLIF(trim(regexp_extract(DOSE, '^[0-9]+([.,][0-9]+)? *(.*)$', 2)), '')                        AS dose_unidade,
    FREQUENCIA_DOSE                                   AS frequencia_dose,
    POSOLOGIA                                         AS posologia,

    DURACAO                                           AS duracao_texto,
    anvisa.fn_duracao_em_dias(DURACAO)                AS duracao_dias,

    anvisa.fn_parse_data(INICIO_ADMINISTRACAO, TRUE)  AS data_inicio_administracao,
    anvisa.fn_data_incompleta(INICIO_ADMINISTRACAO)   AS ind_inicio_incompleta,
    anvisa.fn_parse_data(FIM_ADMINISTRACAO, TRUE)     AS data_fim_administracao,
    anvisa.fn_data_incompleta(FIM_ADMINISTRACAO)      AS ind_fim_incompleta,

    FORMA_FARMACEUTICA                                AS forma_farmaceutica,
    VIA_ADMINISTRACAO                                 AS via_administracao,
    VIA_ADMINISTRACAO_MAE_PAI                         AS via_administracao_mae_pai,
    NUMERO_LOTE                                       AS numero_lote,
    _data_ingestao                                    AS dt_ingestao_bronze
  FROM filtrado
),

/* ---------------------------------------------------------------------
   4. regras  -  PLAUSIBILIDADE
   - Data de administracao no futuro -> NULL.
   - Fim anterior ao inicio: nao apaga; so sinaliza.
   - Dose <= 0 nao faz sentido -> NULL (a unidade permanece).
   --------------------------------------------------------------------- */
regras AS (
  SELECT
    * EXCEPT (data_inicio_administracao, data_fim_administracao, dose_valor),
    CASE WHEN data_inicio_administracao <= current_date() THEN data_inicio_administracao END AS data_inicio_administracao,
    CASE WHEN data_fim_administracao    <= current_date() THEN data_fim_administracao    END AS data_fim_administracao,
    CASE WHEN dose_valor > 0 THEN dose_valor END                                             AS dose_valor
  FROM tipado
),

final AS (
  SELECT
    id_notificacao, relacao_medicamento_evento, nome_medicamento,
    principios_ativos, codigos_atc, detentor_registro, concentracao,
    componente_suspeito, acao_adotada, problemas_adicionais,
    indicacao_meddra, indicacao_relatada,
    dose_texto, dose_valor, dose_unidade, frequencia_dose, posologia,
    duracao_texto, duracao_dias,
    data_inicio_administracao, ind_inicio_incompleta,
    data_fim_administracao,    ind_fim_incompleta,
    coalesce(data_fim_administracao < data_inicio_administracao, FALSE) AS ind_fim_anterior_inicio,
    forma_farmaceutica, via_administracao, via_administracao_mae_pai, numero_lote,
    dt_ingestao_bronze,
    current_timestamp() AS dt_processamento_silver
  FROM regras
)

/* DEDUPLICACAO: remove so duplicatas exatas (todas as colunas iguais).
   O mesmo medicamento pode aparecer em linhas distintas da mesma notificacao
   (doses ou periodos diferentes) e isso e informacao legitima. */
SELECT DISTINCT * FROM final;

num_affected_rows,num_inserted_rows


### 2.4 s_dados_abertos_medicamentos

In [0]:
/* =====================================================================
   2.4 - s_dados_abertos_medicamentos
   Origem : anvisa.b_dados_abertos_medicamentos   (Bronze, tudo STRING)
   Destino: anvisa.s_dados_abertos_medicamentos   (Silver, tipada)
   GRAO   : 1 linha por registro do arquivo de Dados Abertos (catalogo de
            medicamentos da Anvisa). O mesmo numero de registro pode repetir
            (varios principios ativos/apresentacoes), por isso NAO e chave.

   Etapas (CTEs): limpo -> filtrado -> tipado -> regras -> final
   ===================================================================== */

CREATE OR REPLACE TABLE anvisa.s_dados_abertos_medicamentos
USING DELTA
-- LOCATION 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/02_silver/s_dados_abertos_medicamentos'
COMMENT 'Silver Anvisa Dados Abertos: catalogo de medicamentos registrados, tipado e sem duplicatas exatas.'
AS
WITH

/* ---------------------------------------------------------------------
   1. limpo   ('None'/vazio -> NULL; espacos duplicados colapsados)
   --------------------------------------------------------------------- */
limpo AS (
  SELECT
    upper(anvisa.fn_limpa_texto(TIPO_PRODUTO))      AS TIPO_PRODUTO,
    anvisa.fn_limpa_texto(NOME_PRODUTO)             AS NOME_PRODUTO,
    DATA_FINALIZACAO_PROCESSO,
    anvisa.fn_limpa_texto(CATEGORIA_REGULATORIA)    AS CATEGORIA_REGULATORIA,
    anvisa.fn_limpa_texto(NUMERO_REGISTRO_PRODUTO)  AS NUMERO_REGISTRO_PRODUTO,
    DATA_VENCIMENTO_REGISTRO,
    anvisa.fn_limpa_texto(NUMERO_PROCESSO)          AS NUMERO_PROCESSO,
    anvisa.fn_limpa_texto(CLASSE_TERAPEUTICA)       AS CLASSE_TERAPEUTICA,
    anvisa.fn_limpa_texto(EMPRESA_DETENTORA_REGISTRO) AS EMPRESA_DETENTORA_REGISTRO,
    anvisa.fn_limpa_texto(SITUACAO_REGISTRO)        AS SITUACAO_REGISTRO,
    anvisa.fn_limpa_texto(PRINCIPIO_ATIVO)          AS PRINCIPIO_ATIVO,
    _data_ingestao
  FROM anvisa.b_dados_abertos_medicamentos
),

/* ---------------------------------------------------------------------
   2. filtrado
   REGRA: a Bronze tem 1 linha sem NOME_PRODUTO, empresa e situacao (linha
   quebrada do CSV). Sem nome do produto nao ha o que catalogar -> DESCARTADA.
   Linhas SEM numero de registro (~21 mil) sao MANTIDAS: nao sabemos se sao
   falha de qualidade ou produtos sem registro numerico; descartar poderia
   apagar produtos validos. Elas ficam identificaveis por
   numero_registro_produto IS NULL (veja a secao 3).
   --------------------------------------------------------------------- */
filtrado AS (
  SELECT *
  FROM limpo
  WHERE NOME_PRODUTO IS NOT NULL
),

/* ---------------------------------------------------------------------
   3. tipado
   REGRAS
   - numero_registro_produto e numero_processo: mantidos como STRING (sao
     identificadores, nao quantidades) contendo so digitos. Nunca como INT,
     para nao perder zeros a esquerda nem estourar o tamanho.
   - EMPRESA_DETENTORA_REGISTRO vem como '33173097000274 - CELLERA ...':
     separada em cnpj (14 digitos) e nome da empresa.
   - data_finalizacao_processo vem como dd/MM/yyyy -> prioridade BR.
   - data_vencimento_registro vem como MMyyyy (ex.: '042029'). Nao ha dia,
     entao e gravada como DIA 01 do mes (2029-04-01). Interprete como "o
     registro vence NO MES de abril/2029", nao exatamente no dia 1.
   - situacao_registro -> tambem como BOOLEAN (ind_registro_ativo).
   - principio_ativo: texto livre com virgulas legitimas no nome
     (ex.: 'cloridrato de ondansetrona di-hidratado, cloridrato de ...'),
     por isso NAO e quebrado em array; so e padronizado em minusculas.
   --------------------------------------------------------------------- */
tipado AS (
  SELECT
    TIPO_PRODUTO                                                         AS tipo_produto,
    NOME_PRODUTO                                                         AS nome_produto,
    CATEGORIA_REGULATORIA                                                AS categoria_regulatoria,
    NULLIF(regexp_replace(NUMERO_REGISTRO_PRODUTO, '[^0-9]', ''), '')    AS numero_registro_produto,
    NULLIF(regexp_replace(NUMERO_PROCESSO, '[^0-9]', ''), '')            AS numero_processo,
    CLASSE_TERAPEUTICA                                                   AS classe_terapeutica,
    NULLIF(regexp_extract(EMPRESA_DETENTORA_REGISTRO, '^([0-9]{14}) *- *(.*)$', 1), '')   AS cnpj_empresa_detentora,
    coalesce(NULLIF(regexp_extract(EMPRESA_DETENTORA_REGISTRO, '^([0-9]{14}) *- *(.*)$', 2), ''),
             EMPRESA_DETENTORA_REGISTRO)                                 AS empresa_detentora_registro,
    SITUACAO_REGISTRO                                                    AS situacao_registro,
    CASE lower(SITUACAO_REGISTRO) WHEN 'ativo'   THEN TRUE
                                  WHEN 'inativo' THEN FALSE END          AS ind_registro_ativo,
    lower(PRINCIPIO_ATIVO)                                               AS principio_ativo,
    anvisa.fn_parse_data(DATA_FINALIZACAO_PROCESSO, TRUE)                AS data_finalizacao_processo,
    anvisa.fn_parse_data(DATA_VENCIMENTO_REGISTRO,  TRUE)                AS data_vencimento_registro,
    anvisa.fn_data_incompleta(DATA_VENCIMENTO_REGISTRO)                  AS ind_vencimento_so_mes,
    _data_ingestao                                                       AS dt_ingestao_bronze
  FROM filtrado
),

/* ---------------------------------------------------------------------
   4. regras  -  PLAUSIBILIDADE
   - Finalizacao de processo no futuro e impossivel -> NULL.
   - Vencimento de registro PODE ser futuro (e o normal): nao filtrar.
   --------------------------------------------------------------------- */
final AS (
  SELECT
    tipo_produto, nome_produto, categoria_regulatoria,
    numero_registro_produto, numero_processo, classe_terapeutica,
    cnpj_empresa_detentora, empresa_detentora_registro,
    situacao_registro, ind_registro_ativo, principio_ativo,
    CASE WHEN data_finalizacao_processo <= current_date() THEN data_finalizacao_processo END AS data_finalizacao_processo,
    data_vencimento_registro, ind_vencimento_so_mes,
    dt_ingestao_bronze,
    current_timestamp() AS dt_processamento_silver
  FROM tipado
)

/* DEDUPLICACAO: remove so duplicatas exatas. Linhas com o mesmo registro mas
   principio ativo/apresentacao diferente sao registros distintos. */
SELECT DISTINCT * FROM final;

num_affected_rows,num_inserted_rows


## 3. Validacoes
Cada celula e independente.

In [0]:
-- ---------------------------------------------------------------------
-- A) TESTE UNITARIO DAS FUNCOES DE DATA  (rodar apos criar as funcoes)
--    Compare a coluna "esperado" com os resultados.
-- ---------------------------------------------------------------------
SELECT
  entrada,
  anvisa.fn_parse_data(entrada, TRUE)   AS resultado_prioridade_br,
  anvisa.fn_parse_data(entrada, FALSE)  AS resultado_prioridade_us,
  anvisa.fn_data_incompleta(entrada)    AS incompleta,
  esperado
FROM VALUES
  ('09/28/2023 00:00:00.000000', '2023-09-28 (qualquer prioridade)'),
  ('28/09/2023',                 '2023-09-28 (qualquer prioridade)'),
  ('03/04/2023',                 'BR: 2023-04-03 | US: 2023-03-04'),
  ('20181112',                   '2018-11-12'),
  ('2018-11-12',                 '2018-11-12'),
  ('2018-11-12T10:30:00',        '2018-11-12'),
  ('201811121030',               '2018-11-12'),
  ('042029',                     '2029-04-01, incompleta = true'),
  ('202903',                     '2029-03-01, incompleta = true'),
  ('11/2018',                    '2018-11-01, incompleta = true'),
  ('2018-11',                    '2018-11-01, incompleta = true'),
  ('2018',                       '2018-01-01, incompleta = true'),
  ('15/03',                      'NULL (sem ano)'),
  ('31/02/2020',                 'NULL (data impossivel)'),
  ('None',                       'NULL'),
  ('',                           'NULL'),
  (NULL,                         'NULL')
AS t(entrada, esperado);

entrada,resultado_prioridade_br,resultado_prioridade_us,incompleta,esperado
09/28/2023 00:00:00.000000,2023-09-28,2023-09-28,false,2023-09-28 (qualquer prioridade)
28/09/2023,2023-09-28,2023-09-28,false,2023-09-28 (qualquer prioridade)
03/04/2023,2023-04-03,2023-03-04,false,BR: 2023-04-03 | US: 2023-03-04
20181112,2018-11-12,2018-11-12,false,2018-11-12
2018-11-12,2018-11-12,2018-11-12,false,2018-11-12
2018-11-12T10:30:00,2018-11-12,2018-11-12,false,2018-11-12
201811121030,2018-11-12,2018-11-12,false,2018-11-12
042029,2029-04-01,2029-04-01,true,"2029-04-01, incompleta = true"
202903,2029-03-01,2029-03-01,true,"2029-03-01, incompleta = true"
11/2018,2018-11-01,2018-11-01,true,"2018-11-01, incompleta = true"


In [0]:
-- Teste de duracao, texto e Sim/Nao
SELECT 'duracao' AS teste, '5 minuto' AS entrada, anvisa.fn_duracao_em_dias('5 minuto') AS resultado, '0.0035' AS esperado
UNION ALL SELECT 'duracao', '3 dia',  anvisa.fn_duracao_em_dias('3 dia'),  '3'
UNION ALL SELECT 'duracao', '2 semana', anvisa.fn_duracao_em_dias('2 semana'), '14'
UNION ALL SELECT 'duracao', 'None',   anvisa.fn_duracao_em_dias('None'),   'NULL'
UNION ALL SELECT 'texto',   ' None ', anvisa.fn_limpa_texto(' None '),     'NULL'
UNION ALL SELECT 'sim_nao', 'Não ',   CAST(anvisa.fn_sim_nao('Não ') AS STRING), 'false';

In [0]:
-- ---------------------------------------------------------------------
-- B) VOLUMETRIA: Bronze x Silver e linhas descartadas
-- ---------------------------------------------------------------------
SELECT 'notificacoes' AS tabela,
       (SELECT count(*) FROM anvisa.b_vigimed_notificacoes)               AS linhas_bronze,
       (SELECT count(*) FROM anvisa.s_vigimed_notificacoes)               AS linhas_silver
UNION ALL
SELECT 'reacoes',
       (SELECT count(*) FROM anvisa.b_vigimed_reacoes),
       (SELECT count(*) FROM anvisa.s_vigimed_reacoes)
UNION ALL
SELECT 'medicamentos',
       (SELECT count(*) FROM anvisa.b_vigimed_medicamentos),
       (SELECT count(*) FROM anvisa.s_vigimed_medicamentos)
UNION ALL
SELECT 'dados_abertos_medicamentos',
       (SELECT count(*) FROM anvisa.b_dados_abertos_medicamentos),
       (SELECT count(*) FROM anvisa.s_dados_abertos_medicamentos);

tabela,linhas_bronze,linhas_silver
notificacoes,396495,363249
reacoes,1114975,1027408
medicamentos,711105,688856
dados_abertos_medicamentos,229611,42741


In [0]:
-- ---------------------------------------------------------------------
-- C) UNICIDADE: id_notificacao deve ser unico na Silver de notificacoes
--    Resultado esperado: 0 linhas
-- ---------------------------------------------------------------------
SELECT id_notificacao, count(*) AS qtd
FROM anvisa.s_vigimed_notificacoes
GROUP BY id_notificacao
HAVING count(*) > 1;

id_notificacao,qtd


In [0]:
-- ---------------------------------------------------------------------
-- D) DATAS QUE NAO FORAM INTERPRETADAS
--    Mostra valores da Bronze que tinham conteudo mas viraram NULL na Silver.
--    Se aparecerem formatos novos, ajuste fn_parse_data (secao 1).
-- ---------------------------------------------------------------------
SELECT 'notificacoes.DATA_INCLUSAO_SISTEMA' AS coluna, DATA_INCLUSAO_SISTEMA AS valor_bronze, count(*) AS qtd
FROM anvisa.b_vigimed_notificacoes
WHERE anvisa.fn_limpa_texto(DATA_INCLUSAO_SISTEMA) IS NOT NULL
  AND anvisa.fn_parse_data(DATA_INCLUSAO_SISTEMA, FALSE) IS NULL
GROUP BY 1, 2
UNION ALL
SELECT 'notificacoes.DATA_NASCIMENTO', DATA_NASCIMENTO, count(*)
FROM anvisa.b_vigimed_notificacoes
WHERE anvisa.fn_limpa_texto(DATA_NASCIMENTO) IS NOT NULL
  AND anvisa.fn_parse_data(DATA_NASCIMENTO, FALSE) IS NULL
GROUP BY 1, 2
UNION ALL
SELECT 'reacoes.DATA_INICIO_HORA', DATA_INICIO_HORA, count(*)
FROM anvisa.b_vigimed_reacoes
WHERE anvisa.fn_limpa_texto(DATA_INICIO_HORA) IS NOT NULL
  AND anvisa.fn_parse_data(DATA_INICIO_HORA, TRUE) IS NULL
GROUP BY 1, 2
UNION ALL
SELECT 'reacoes.DATA_FINAL_HORA', DATA_FINAL_HORA, count(*)
FROM anvisa.b_vigimed_reacoes
WHERE anvisa.fn_limpa_texto(DATA_FINAL_HORA) IS NOT NULL
  AND anvisa.fn_parse_data(DATA_FINAL_HORA, TRUE) IS NULL
GROUP BY 1, 2
UNION ALL
SELECT 'medicamentos.INICIO_ADMINISTRACAO', INICIO_ADMINISTRACAO, count(*)
FROM anvisa.b_vigimed_medicamentos
WHERE anvisa.fn_limpa_texto(INICIO_ADMINISTRACAO) IS NOT NULL
  AND anvisa.fn_parse_data(INICIO_ADMINISTRACAO, TRUE) IS NULL
GROUP BY 1, 2
UNION ALL
SELECT 'dados_abertos.DATA_FINALIZACAO_PROCESSO', DATA_FINALIZACAO_PROCESSO, count(*)
FROM anvisa.b_dados_abertos_medicamentos
WHERE anvisa.fn_limpa_texto(DATA_FINALIZACAO_PROCESSO) IS NOT NULL
  AND anvisa.fn_parse_data(DATA_FINALIZACAO_PROCESSO, TRUE) IS NULL
GROUP BY 1, 2
UNION ALL
SELECT 'dados_abertos.DATA_VENCIMENTO_REGISTRO', DATA_VENCIMENTO_REGISTRO, count(*)
FROM anvisa.b_dados_abertos_medicamentos
WHERE anvisa.fn_limpa_texto(DATA_VENCIMENTO_REGISTRO) IS NOT NULL
  AND anvisa.fn_parse_data(DATA_VENCIMENTO_REGISTRO, TRUE) IS NULL
GROUP BY 1, 2
ORDER BY qtd DESC
LIMIT 100;

coluna,valor_bronze,qtd
reacoes.DATA_FINAL_HORA,19:00:00,35
dados_abertos.DATA_FINALIZACAO_PROCESSO,14/01/0097,24
dados_abertos.DATA_FINALIZACAO_PROCESSO,01/12/0001,13
dados_abertos.DATA_VENCIMENTO_REGISTRO,09200,9
reacoes.DATA_FINAL_HORA,07:00:00,7
dados_abertos.DATA_FINALIZACAO_PROCESSO,10/11/0084,7
reacoes.DATA_FINAL_HORA,22:00:00,7
reacoes.DATA_INICIO_HORA,14:30:00,5
reacoes.DATA_FINAL_HORA,15:15:00,5
reacoes.DATA_INICIO_HORA,08:00:00,5


In [0]:
-- ---------------------------------------------------------------------
-- E) INTEGRIDADE ENTRE TABELAS (orfaos)
--    Reacoes/medicamentos cujo id nao existe em notificacoes. Esperado: algum
--    volume, porque a Bronze de notificacoes perdeu linhas quebradas do CSV.
-- ---------------------------------------------------------------------
SELECT 'reacoes sem notificacao' AS checagem, count(DISTINCT r.id_notificacao) AS ids
FROM anvisa.s_vigimed_reacoes r
LEFT ANTI JOIN anvisa.s_vigimed_notificacoes n ON n.id_notificacao = r.id_notificacao
UNION ALL
SELECT 'medicamentos sem notificacao', count(DISTINCT m.id_notificacao)
FROM anvisa.s_vigimed_medicamentos m
LEFT ANTI JOIN anvisa.s_vigimed_notificacoes n ON n.id_notificacao = m.id_notificacao
UNION ALL
SELECT 'notificacoes sem reacao', count(*)
FROM anvisa.s_vigimed_notificacoes n
LEFT ANTI JOIN (SELECT DISTINCT id_notificacao FROM anvisa.s_vigimed_reacoes) r USING (id_notificacao);

checagem,ids
reacoes sem notificacao,22303
medicamentos sem notificacao,2940
notificacoes sem reacao,400


In [0]:
-- ---------------------------------------------------------------------
-- F) PERCENTUAL DE NULOS DAS COLUNAS PRINCIPAIS NA SILVER
-- ---------------------------------------------------------------------
SELECT 'notificacoes' AS tabela, count(*) AS linhas,
       round(100 * avg(CAST(uf IS NULL AS INT)), 1)                      AS pct_nulo_uf,
       round(100 * avg(CAST(data_nascimento IS NULL AS INT)), 1)         AS pct_nulo_data_nascimento,
       round(100 * avg(CAST(idade_momento_reacao IS NULL AS INT)), 1)    AS pct_nulo_idade,
       round(100 * avg(CAST(peso_kg IS NULL AS INT)), 1)                 AS pct_nulo_peso,
       round(100 * avg(CAST(altura_cm IS NULL AS INT)), 1)               AS pct_nulo_altura
FROM anvisa.s_vigimed_notificacoes;

tabela,linhas,pct_nulo_uf,pct_nulo_data_nascimento,pct_nulo_idade,pct_nulo_peso,pct_nulo_altura
notificacoes,363249,23.5,26.3,11.5,60.9,78.1


In [0]:
SELECT count(*) AS linhas,
       round(100 * avg(CAST(data_inicio IS NULL AS INT)), 1)                  AS pct_nulo_data_inicio,
       round(100 * avg(CAST(ind_data_inicio_incompleta AS INT)), 1)           AS pct_inicio_incompleta,
       sum(CAST(ind_data_fim_anterior_inicio AS INT))                         AS qtd_fim_antes_inicio
FROM anvisa.s_vigimed_reacoes;

linhas,pct_nulo_data_inicio,pct_inicio_incompleta,qtd_fim_antes_inicio
1027408,44.2,17.8,6028


In [0]:
%python
import json
from pyspark.sql import functions as F

def gerar_perfil_tabela(nome_tabela):
    print(f"Iniciando o perfil da tabela: {nome_tabela}")
    df = spark.table(nome_tabela)

    # 1. Agregação eficiente dos valores distintos e nulos (dispara apenas 1 Job)
    exprs = []
    for c in df.columns:
        exprs.append(F.approx_count_distinct(F.col(c)).alias(f"{c}_distinct"))
        exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls"))

    agg_results = df.agg(*exprs).collect()[0].asDict()

    # 2. Montar dicionário estruturado
    perfil_tabela = {
        "tabela": nome_tabela,
        "dicionario_de_dados": [
            {
                "coluna": col,
                "tipo_de_dado": dtype,
                "valores_distintos_aprox": agg_results[f"{col}_distinct"],
                "quantidade_nulos": agg_results[f"{col}_nulls"]
            }
            for col, dtype in df.dtypes
        ],
        "amostra_dados": [row.asDict() for row in df.limit(5).collect()]
    }

    # 3. Formatar em JSON
    # O uso de default=str é vital para evitar erros com datas ou decimais do PySpark
    resultado_llm = json.dumps(perfil_tabela, indent=2, default=str, ensure_ascii=False)

    # 4. Salvar o resultado em um arquivo JSON dinâmico
    # Substituímos pontos por underlines para evitar problemas com o nome do arquivo
    nome_arquivo_seguro = nome_tabela.replace(".", "_")
    nome_arquivo = f"{nome_arquivo_seguro}_schema.json"
    
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write(resultado_llm)

    print(f"Arquivo '{nome_arquivo}' salvo com sucesso!\n")

# Lista de tabelas que você deseja processar
tabelas = [
    "anvisa.s_dados_abertos_medicamentos",
    "anvisa.s_vigimed_medicamentos",
    "anvisa.s_vigimed_notificacoes",
    "anvisa.s_vigimed_reacoes"
]

# Executa a função para cada tabela na lista
for tb in tabelas:
    gerar_perfil_tabela(tb)

Iniciando o perfil da tabela: anvisa.s_dados_abertos_medicamentos
Arquivo 'anvisa_s_dados_abertos_medicamentos_schema.json' salvo com sucesso!

Iniciando o perfil da tabela: anvisa.s_vigimed_medicamentos
Arquivo 'anvisa_s_vigimed_medicamentos_schema.json' salvo com sucesso!

Iniciando o perfil da tabela: anvisa.s_vigimed_notificacoes
Arquivo 'anvisa_s_vigimed_notificacoes_schema.json' salvo com sucesso!

Iniciando o perfil da tabela: anvisa.s_vigimed_reacoes
Arquivo 'anvisa_s_vigimed_reacoes_schema.json' salvo com sucesso!

